In [1]:
!pip install tensorflow==2.10
!pip install image-classifiers==0.2.2
!pip install segmentation-models
!pip install psutil

ERROR: Could not find a version that satisfies the requirement tensorflow==2.10 (from versions: 2.12.0rc0, 2.12.0rc1, 2.12.0, 2.12.1, 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.10


  Using cached image_classifiers-0.2.2-py2.py3-none-any.whl.metadata (8.3 kB)
Using cached image_classifiers-0.2.2-py2.py3-none-any.whl (72 kB)
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 1.0.0
    Uninstalling image-classifiers-1.0.0:
      Successfully uninstalled image-classifiers-1.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
segmentation-models 1.0.1 requires image-classifiers==1.0.0, but you have image-classifiers 0.2.2 which is incompatible.


  Using cached image_classifiers-1.0.0-py3-none-any.whl.metadata (8.6 kB)
Using cached image_classifiers-1.0.0-py3-none-any.whl (19 kB)
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 0.2.2
    Uninstalling image-classifiers-0.2.2:
      Successfully uninstalled image-classifiers-0.2.2


In [2]:
import os
import json
import csv
import time
import psutil
from datetime import datetime
from classification_models.keras import Classifiers
from tensorflow import keras
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    CSVLogger,
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
)
import subprocess
import sys
sys.path.append("C:/Users/sophi/OneDrive/Documents/TH/Bachelorarbeit/Code/notebooks/SISA/sisa-fork")
from datasets.CIFAR10.dataloader_all_data import load
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000


TODO:
- Model laden + die Pytorch Impl anpassen -> DONE
- Datensaetze vorbereiten -> DONE
- Korrekte Argumente für Skripte
- Validationdatensatz in Training
- Extra Eval fuer Vergleichbarkeit
- Ausfuerhung in Colab moeglich machen -> Ordnerstruktur

In [3]:
#Global Config
BATCH_SIZE = 64 #more balance for gradient updates
EPOCHS = 30
LEARNING_RATE = 0.001
IMAGE_SIZE = 32 # CIFAR-10 is 32x32, better resolution than with 64x64 ( no resizing noise)
NUM_CLASSES = 10

#initializes NumpY and Tenserflow to work deterministicaly
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

DATAFILEPATH= r"datasets/CIFAR10/datasetfile_all_data"
DATALOADER= "datasets.CIFAR10.dataloader_all_data"
OPTIMIZER = "adam" # --optimizer adam

#SISA Specific
CONTAINER_NAME= "Baseline"
NUM_SHARDS = 10
NUM_SLICES = 5
MODEL_NAME="resnet18-pytorch"
LABEL="0"

In [4]:
#Log what GPU is used in training -> Colab specific
!nvidia-smi

Der Befehl "nvidia-smi" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [5]:
BASE_RUN_DIR = "experiments"


def create_run():
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_path = os.path.join(BASE_RUN_DIR, run_id)

    os.makedirs(run_path, exist_ok=True)

    return run_id, run_path



def save_config(run_path, config):
    with open(os.path.join(run_path, "config.json"), "w") as f:
        json.dump(config, f, indent=4)

def save_metrics(run_path, metrics, filename="metrics.json"):
    serializable_metrics = {
        k: v.tolist() if isinstance(v, np.ndarray) else v
        for k, v in metrics.items()
    }

    with open(os.path.join(run_path, filename), "w") as f:
        json.dump(serializable_metrics, f, indent=4)

In [6]:
# Daten laden (aus Baseline-Dataloader)
#Preproccesing already done in dataloader
from datasets.CIFAR10.dataloader_all_data import x_train, y_train, x_val, y_val, x_test, y_test

In [7]:
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

print("x_val shape:", x_val.shape)
print("y_val shape:", y_val.shape)

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

print("Number of training datapoints:", x_train.shape[0])
print("Number of validation datapoints:", x_val.shape[0])
print("Number of test datapoints:", x_test.shape[0])

x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000


In [8]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(50000, seed=SEED, reshuffle_each_iteration=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds=tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds =val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds =test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
print("\n=== TRAIN DATASET after Prepeocessing/Data Augmentation) ===")

for x_batch, y_batch in train_ds.take(1):
    print("Train batch x shape:", x_batch.shape)
    print("Train batch y shape:", y_batch.shape)

print("\n=== TEST DATASET after Preprocessing ===")

for x_batch, y_batch in test_ds.take(1):
    print("Test batch x shape:", x_batch.shape)
    print("Test batch y shape:", y_batch.shape)

train_size = tf.data.experimental.cardinality(train_ds).numpy() * BATCH_SIZE
test_size = tf.data.experimental.cardinality(test_ds).numpy() * BATCH_SIZE

print("\n=== DATASET SIZE CHECK ===")
print("Approx train samples:", train_size)
print("Approx test samples:", test_size)

#did preprocessing/augmentation work?
print("\n=== SAMPLE TRANSFORM CHECK ===")

for x_batch, y_batch in train_ds.take(1):
    print("Min pixel value:", tf.reduce_min(x_batch).numpy())
    print("Max pixel value:", tf.reduce_max(x_batch).numpy())
    print("Mean pixel value:", tf.reduce_mean(x_batch).numpy())



=== TRAIN DATASET after Prepeocessing/Data Augmentation) ===
Train batch x shape: (64, 32, 32, 3)
Train batch y shape: (64, 1)

=== TEST DATASET after Preprocessing ===
Test batch x shape: (64, 32, 32, 3)
Test batch y shape: (64, 1)

=== DATASET SIZE CHECK ===
Approx train samples: 45056
Approx test samples: 10048

=== SAMPLE TRANSFORM CHECK ===
Min pixel value: -1.0
Max pixel value: 1.0
Mean pixel value: 0.042378526


In [10]:
#Training

In [11]:
#Ordnerstrukur für Conatiner erstellen
container_path = f"containers/{CONTAINER_NAME}"
os.makedirs(f"{container_path}/cache", exist_ok=True)
os.makedirs(f"{container_path}/times", exist_ok=True)
os.makedirs(f"{container_path}/outputs", exist_ok=True)

In [13]:
#Initialisierung
init=subprocess.run([
    "python", "../sisa-fork/distribution.py",
    "--shards",str(NUM_SHARDS),
    "--container", CONTAINER_NAME,
    "--dataset", DATAFILEPATH,
    "--label", LABEL
],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
print("STDOUT:", init.stdout)
print("STDERR:", init.stderr)
print("Returncode:", init.returncode)

STDOUT: 
STDERR: 
Returncode: 0


In [14]:
# Überprüfung splitfile.npy
splitfile_path = f"{container_path}/splitfile.npy"
if os.path.exists(splitfile_path):
    shards = np.load(splitfile_path, allow_pickle=True)
    print(f"\nInhalt von splitfile.npy (Anzahl der Shards: {len(shards)}):")
    for i, shard in enumerate(shards):
        print(f"  Shard {i}: {len(shard)} Samples")


Inhalt von splitfile.npy (Anzahl der Shards: 10):
  Shard 0: 5000 Samples
  Shard 1: 5000 Samples
  Shard 2: 5000 Samples
  Shard 3: 5000 Samples
  Shard 4: 5000 Samples
  Shard 5: 5000 Samples
  Shard 6: 5000 Samples
  Shard 7: 5000 Samples
  Shard 8: 5000 Samples
  Shard 9: 5000 Samples


In [15]:
# Überprüfung requestfile:0.npy
requestfile_path = f"{container_path}/requestfile:0.npy"
if os.path.exists(requestfile_path):
    requests = np.load(requestfile_path, allow_pickle=True)
    print(f"\nInhalt von requestfile:0.npy (Anzahl der Shards: {len(requests)}):")
    for i, req in enumerate(requests):
        print(f"  Shard {i}: {len(req)} Requests")


Inhalt von requestfile:0.npy (Anzahl der Shards: 10):
  Shard 0: 0 Requests
  Shard 1: 0 Requests
  Shard 2: 0 Requests
  Shard 3: 0 Requests
  Shard 4: 0 Requests
  Shard 5: 0 Requests
  Shard 6: 0 Requests
  Shard 7: 0 Requests
  Shard 8: 0 Requests
  Shard 9: 0 Requests


In [73]:
#Jetzt 2 dateien erstellt
#splitfile.npy:  Eine Liste vonEine Liste von NumPy-Arrays, wobei jedes Array die Indizes der zu löschenden Daten für einen Shard enthält NumPy-Arrays, wobei jedes Array die Indizes der Trainingsdaten für einen Shard enthält
#requestfile:<label>.npy: Eine Liste von NumPy-Arrays, wobei jedes Array die Indizes der zu löschenden Daten für einen Shard enthält -> Hier Initiales Training,also Datei leer

In [74]:
# Training
#alle Arumente als Strings, da subprocess das so braucht
for i in range(NUM_SHARDS):
    print(f"Training Shard {i+1}/{NUM_SHARDS}...")
    train=subprocess.run([
        "python", "../sisa/sisa-fork/sisa.py",
        "--model", MODEL_NAME,
        "--train",
        "--slices", str(NUM_SLICES),
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
print(f"Ouput of Traing shard {i+1}/{NUM_SHARDS}")
print("STDOUT:", train.stdout)
print("STDERR:", train.stderr)
print("Returncode:", train.returncode)

Training Shard 1/10...
Training Shard 2/10...
Training Shard 3/10...
Training Shard 4/10...
Training Shard 5/10...
Training Shard 6/10...
Training Shard 7/10...
Training Shard 8/10...
Training Shard 9/10...
Training Shard 10/10...
Ouput of Traing shard 10/10
STDOUT: 
STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Traceback (most recent call last):
  File "C:\Users\sophi\OneDrive\Documents\TH\Bachelorarbeit\Code\notebooks\sisa\sisa-fork\sisa.py", line 186, in <module>
    for images, labels in fetchShardBatch(
  File "C:\Users\sophi\OneDrive\Documents\TH\Bachelorarbeit\Code\notebooks\sisa\sisa-fork\sharded.py", line 48, in fetchShardBatch
    dataloader = importlib.import_module('.'.join(dataset.split('/')[:-1] + [datasetfile

In [16]:
#Prüfen ob Training funktioniert hat
result=subprocess.run([
    "python", r"../sisa-fork/sisa.py",  # Rohstring für den Pfad
     "--model", MODEL_NAME,
        "--train",
        "--slices", str(NUM_SLICES),
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", "0"
],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
# Gib die Standardausgabe und Fehlerausgabe aus
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Returncode:", result.returncode)

STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000

STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
I0000 00:00:1782391914.037856   25656 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782391922.412848   25656 port.cc:153] oneDNN custom operations are on. You may see slightly different nu

In [ ]:
# Vorhersagen
for i in range(NUM_SHARDS):
    print(f"Predicting with Shard {i+1}/{NUM_SHARDS}...")
    subprocess.run([
        "python", "../sisa/sisa.py",
        "--model", model_name,
        "--test", #Important
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--batch_size", BATCH_SIZE,
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ])

In [ ]:
#7. Aggregation 
from sisa.aggregation import main as aggregation_main
import argparse

args_agg = argparse.Namespace(strategy="uniform", CONTAINER_NAME, shards=NUM_SHARDS,
                             dataset=DATAFILEPATH, label=LABEL)
accuracy = aggregation_main(args_agg)

In [ ]:
# Lade Vorhersagen aller Shards
outputs = []
for i in range(shards):
    pred_file = f"../sisa/containers/{CONTAINER_NAME}/outputs/shard-{i}:{label}.npy"
    pred = np.load(pred_file)
    outputs.append(pred)
outputs = np.array(outputs)

In [ ]:
#TODO: get the general-report.csv for results 

In [ ]:
# Unlearning

In [ ]:
# Generiere Requests
subprocess.run([
    "python", "../sisa/distribution.py",
    "--requests", "100",
    "--distribution", "uniform",
    "--container", CONTAINER_NAME,
    "--dataset", DATAFILEPATH,
    "--label", "1"
])


In [ ]:
# Trainiere betroffenen Shards neu
for i in range(shards):
    print(f"Retraining Shard {i+1}/{shards} with Unlearning...")
    subprocess.run([
        "python", "../sisa/sisa.py",
        "--model", model_name,
        "--train",
        "--slices", "1",
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", EPOCHS,
        "--batch_size", BATCH_SIZE,
        "--learning_rate", LEARNING_RATE,
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ])


In [ ]:
# Berechne Vorhersagen für Unlearning
for i in range(shards):
    subprocess.run([
        "python", "../sisa/sisa.py",
        "--model", model_name,
        "--test",
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--batch_size", BATCH_SIZE,
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ])

In [ ]:
# Aggregiere Vorhersagen
args_agg = argparse.Namespace(strategy="uniform", container=CONTAINER_NAME, shards=NUM_SHARDS,
                             dataset=dataset_path_unlearning, label="1")
accuracy_unlearn = aggregation_main(args_agg)
print(f"Accuracy after Unlearning: {accuracy_unlearn}")

In [ ]:
#TODO extra Eval wie in Baselines fuer Vergleichbarkeit